# Dashboard สินค้าตีกลับ ปี 2569 — รวมข้อมูลรายเดือน

รันทีละเซลล์จากบนลงล่าง (กด ▶ ที่มุมซ้ายของแต่ละเซลล์ หรือกด `Shift+Enter`)
ไม่ต้องใช้ terminal และไม่ต้องติดตั้งอะไรในเครื่องตัวเอง — ทุกอย่างรันบน Google ผ่านเบราว์เซอร์

**ก่อนเริ่ม ต้องมี:**
1. Service account key (ไฟล์ `.json`) ของ `glory-sheets-reader-456@ptglory-dashboard-sales-fb.iam.gserviceaccount.com`
   — ถ้ายังไม่มีไฟล์นี้ ให้ไปที่ Google Cloud Console → IAM & Admin → Service Accounts →
   เลือก account นี้ → แท็บ Keys → Add Key → Create new key → เลือก JSON → กด Create
   (ไฟล์จะดาวน์โหลดลงเครื่องอัตโนมัติ)
2. ทุกไฟล์ Google Sheets รายเดือนต้องแชร์ให้อีเมลนี้เป็น **Viewer** แล้ว (คุณยืนยันแล้วว่าทำแล้ว ✅)


## 1. ติดตั้งไลบรารีที่ต้องใช้
รันเซลล์นี้ก่อน (ใช้เวลาประมาณ 10-20 วินาที)

In [ ]:
!pip install -q gspread pandas google-cloud-bigquery
print("ติดตั้งเสร็จแล้ว")


## 2. อัปโหลด service account key

รันเซลล์ด้านล่าง จะมีปุ่ม **Choose Files** ขึ้นมา ให้เลือกไฟล์ `.json` ของ service account ที่ดาวน์โหลดไว้
(ไฟล์นี้จะอยู่แค่ใน session ของ Colab นี้เท่านั้น หายไปเมื่อปิดแท็บ ไม่ถูกบันทึกที่ไหน)

In [ ]:
from google.colab import files
uploaded = files.upload()
CREDS_PATH = next(iter(uploaded))
print("ใช้ไฟล์:", CREDS_PATH)


## 3. ตั้งค่าไฟล์ต้นทางรายเดือน
รายชื่อไฟล์ + tab (gid) ที่ยืนยันแล้วจากคุณ — ถ้ามีไฟล์เดือนใหม่เพิ่ม ให้เติมแถวใหม่ในรูปแบบเดียวกัน

In [ ]:
SOURCE_SHEETS = [
    {"month": "2026-01", "label": "ม.ค.69", "spreadsheet_id": "1r_oz8FHT4QYe8W6evAjAx2QzmvPuQyRv2Xs5cPtPL6c", "gid": 1814183266, "schema": "A"},
    {"month": "2026-02", "label": "ก.พ.69", "spreadsheet_id": "1qyUUsGSrm6M3Mw6BPuo4aGbGjJJ7fqA-I_w085fDk-c", "gid": 245342036, "schema": "A"},
    {"month": "2026-03", "label": "มี.ค.69", "spreadsheet_id": "1zi-GX6P37N351RR0X-96nPHY1Af44o5jjPQ6UrMoh_A", "gid": 1908257606, "schema": "A"},
    {"month": "2026-04", "label": "เม.ย.69", "spreadsheet_id": "1WWitrg5JbtF9tdw_oLc45USGtQLao1RqW6IbqFSHpoE", "gid": 57782583, "schema": "A"},
    {"month": "2026-05", "label": "พ.ค.69", "spreadsheet_id": "1MlHoMACoJO4xieS-ctKkMt1KfilATfwNg4UZcJRE0H8", "gid": 1451873012, "schema": "A"},
    {"month": "2026-06", "label": "มิ.ย.69", "spreadsheet_id": "1DegHkVMAeYEMftJTRZ7VIZXHNQIbEtDgrNIIoWNC54w", "gid": 1885419356, "schema": "A"},
    {"month": "2026-07", "label": "ก.ค.69", "spreadsheet_id": "1EIKRt4EHeYKPoymw53F5UsxZZTYATPtUd0HCDIwLNtE", "gid": 0, "schema": "B"},
    # 2026-08 (ส.ค.69) ยังไม่มีไฟล์รวมรายเดือน — เพิ่มแถวใหม่ตรงนี้เมื่อรวมไฟล์เสร็จ
]

LOCAL_OUTPUT_CSV = "returns_2569_combined.csv"

# ตั้งเป็น True เมื่อสร้าง BigQuery project/dataset จริงแล้วเท่านั้น
ENABLE_BQ_LOAD = False
BQ_PROJECT = "your-gcp-project-id"
BQ_DATASET = "chargeback_dashboard"
BQ_TABLE = "returns_2569"
BQ_WRITE_MODE = "replace"  # "replace" = โหลดทับทั้งหมดทุกครั้ง (เหมาะกับขนาดข้อมูลนี้)


## 4. ฟังก์ชันรวม + ทำความสะอาดข้อมูล
รันเฉยๆ ไม่ต้องแก้อะไร (ย้ายมาจาก `pipeline/normalize.py` ในโค้ดหลัก)

In [ ]:
import re
import pandas as pd

STANDARD_COLUMNS = [
    "month", "source_schema", "unit",
    "internal_order_id", "online_order_id", "order_status",
    "shop", "sales_channel", "salesperson",
    "transport_company", "tracking_no", "shipping_status",
    "order_time", "ship_date",
    "province",
    "product_code", "product_name", "product_price",
    "payment_method", "return_qty", "is_returned",
    "phone",
]

SCHEMA_A_MAP = {
    "internal_order_id": "หมายเลขออเดอร์ภายใน",
    "online_order_id": "หมายเลขคำสั่งซื้อออนไลน์",
    "order_status": "สถานะคำสั่งซื้อ",
    "transport_company": "บริษัทขนส่ง",
    "tracking_no": "เลขพัสดุ",
    "shipping_status": "สถานะขนส่ง",
    "order_time": "เวลาสั่งซื้อ",
    "shop": "ร้านค้า",
    "province": "จังหวัด",
    "salesperson": "พนักงานขาย",
    "ship_date": "วันที่จัดส่ง",
    "sales_channel": "แพลตฟอร์ม",
    "payment_method": "วิธีการชำระเงิน",
    "phone": "เบอร์โทร",
    "return_qty": "จํานวนสินค้าตีกลับ",
    "product_code": "รหัสสินค้า",
    "product_name": "ชื่อสินค้า",
    "product_price": "ราคาสินค้าทั้งหมด",
}

SCHEMA_B_MAP = {
    "online_order_id": "หมายเลขคำสั่งซื้อออนไลน์",
    "internal_order_id": "หมายเลขออเดอร์ภายใน",
    "order_time": "วันสั่งซื้อ",
    "shop": "ชื่อร้าน",
    "sales_channel": "ฝ่ายที่ขาย",
    "product_code": "รหัสสินค้า",
    "product_name": "ชื่อสินค้า",
    "product_price": "ราคาสินค้า",
    "unit": "Unit",
    "payment_method": "วิธีการชำระเงิน",
    "tracking_no": "หมายเลขพัสดุ",
    "ship_date": "วันที่จัดส่ง",
    "province": "จังหวัด",
    "transport_company": "บริษัทขนส่ง",
    "shipping_status": "สถานะขนส่ง",
    "salesperson": "พนักงานขาย",
}

UNIT_RE = re.compile(r"U(\d+)", re.IGNORECASE)


def _clean_str_cols(df):
    obj_cols = df.select_dtypes("object").columns
    df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())
    return df


def _parse_date(series):
    # Schema A dates เป็น ISO (ไม่กำกวม); Schema B dates เป็น dd/mm/yyyy (แบบไทย)
    # dayfirst=True ใช้ได้กับทั้งสองแบบ
    return pd.to_datetime(series, errors="coerce", dayfirst=True)


def _parse_amount(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.replace("บาท", "", regex=False),
        errors="coerce",
    )


def _extract_unit(product_code):
    return product_code.astype(str).str.extract(UNIT_RE, expand=False).apply(
        lambda v: f"U{v}" if pd.notna(v) else None
    )


def normalize_schema_a(raw, month):
    out = pd.DataFrame(index=raw.index)
    for target, source in SCHEMA_A_MAP.items():
        out[target] = raw[source] if source in raw.columns else None
    out["month"] = month
    out["source_schema"] = "A"
    out["unit"] = _extract_unit(out["product_code"])
    out["order_time"] = _parse_date(out["order_time"])
    out["ship_date"] = _parse_date(out["ship_date"])
    out["product_price"] = _parse_amount(out["product_price"])
    out["return_qty"] = pd.to_numeric(out["return_qty"], errors="coerce").fillna(0)
    out["is_returned"] = (out["return_qty"] > 0) | out["shipping_status"].astype(str).str.contains("ตีกลับ", na=False)
    return out.reindex(columns=STANDARD_COLUMNS)


def normalize_schema_b(raw, month):
    out = pd.DataFrame(index=raw.index)
    for target, source in SCHEMA_B_MAP.items():
        out[target] = raw[source] if source in raw.columns else None
    out["month"] = month
    out["source_schema"] = "B"
    out["order_time"] = _parse_date(out["order_time"])
    out["ship_date"] = _parse_date(out["ship_date"])
    out["product_price"] = _parse_amount(out["product_price"])
    out["return_qty"] = 1
    out["is_returned"] = True
    return out.reindex(columns=STANDARD_COLUMNS)


NORMALIZERS = {"A": normalize_schema_a, "B": normalize_schema_b}


def normalize(raw, schema, month):
    raw = _clean_str_cols(raw.copy())
    return NORMALIZERS[schema](raw, month)


def _dedupe_headers(headers):
    seen = {}
    out = []
    for h in headers:
        h = h.strip()
        if h in seen:
            seen[h] += 1
            out.append(f"{h}__{seen[h]}")
        else:
            seen[h] = 0
            out.append(h)
    return out


def read_sheet_raw(gc, spreadsheet_id, gid):
    ws = gc.open_by_key(spreadsheet_id).get_worksheet_by_id(gid)
    values = ws.get_all_values()
    if not values:
        return pd.DataFrame()
    header, rows = _dedupe_headers(values[0]), values[1:]
    df = pd.DataFrame(rows, columns=header)
    return df.dropna(how="all")

print("โหลดฟังก์ชันเรียบร้อย")


## 5. อ่านข้อมูลจริงจากทุกไฟล์ + รวมกัน
ขั้นตอนนี้อาจใช้เวลา 1-3 นาที (แต่ละไฟล์มี 70,000+ แถว)

In [ ]:
import gspread

gc = gspread.service_account(filename=CREDS_PATH)

frames = []
for src in SOURCE_SHEETS:
    print(f"กำลังอ่าน {src['label']} (schema {src['schema']}) ...")
    raw = read_sheet_raw(gc, src["spreadsheet_id"], src["gid"])
    if raw.empty:
        print(f"  คำเตือน: {src['label']} อ่านได้ 0 แถว ข้ามไฟล์นี้")
        continue
    frames.append(normalize(raw, src["schema"], src["month"]))
    print(f"  ได้ {len(raw)} แถว")

if not frames:
    raise SystemExit("อ่านข้อมูลไม่ได้เลยสักไฟล์ — เช็คว่าแชร์ไฟล์ให้ service account แล้วหรือยัง")

combined = pd.concat(frames, ignore_index=True)
print(f"\nรวมทั้งหมด: {len(combined)} แถว จาก {len(frames)} เดือน")


## 6. ดูสรุปผลลัพธ์
จำนวนแถวต่อเดือน + ตัวอย่างข้อมูล 20 แถวแรก เอาไว้ตรวจสอบว่าข้อมูลดูสมเหตุสมผล

In [ ]:
print("จำนวนแถวต่อเดือน:")
print(combined.groupby("month", dropna=False).size())


In [ ]:
combined.head(20)


In [ ]:
print("จำนวนแถวที่เป็นการตีกลับจริง (is_returned=True) ต่อเดือน:")
print(combined[combined["is_returned"]].groupby("month", dropna=False).size())


## 7. บันทึกไฟล์ CSV + ดาวน์โหลดกลับเครื่อง
ได้ไฟล์เดียวรวมทุกเดือน เปิดด้วย Excel/Google Sheets ต่อได้เลย

In [ ]:
combined.to_csv(LOCAL_OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"บันทึกไฟล์: {LOCAL_OUTPUT_CSV}")

from google.colab import files
files.download(LOCAL_OUTPUT_CSV)


## 8. (ทำทีหลังได้) โหลดเข้า BigQuery

ข้ามขั้นตอนนี้ไปก่อนได้ถ้ายังไม่มี BigQuery project — แค่ดูไฟล์ CSV จากขั้นตอนที่ 7 ก็พอสำหรับตอนนี้

เมื่อพร้อมแล้ว: แก้ค่าตัวแปรด้านบน (`ENABLE_BQ_LOAD = True`, ใส่ `BQ_PROJECT` จริง) แล้วรันเซลล์นี้
ต้องรัน authenticate ของ Colab เองด้วย (`from google.colab import auth; auth.authenticate_user()`)
หรือใช้ service account เดิม (แต่ service account ต้องมีสิทธิ์ BigQuery Data Editor + Job User บน project นั้น)

In [ ]:
if ENABLE_BQ_LOAD:
    from google.cloud import bigquery
    from google.cloud.bigquery import LoadJobConfig, WriteDisposition

    client = bigquery.Client.from_service_account_json(CREDS_PATH, project=BQ_PROJECT)
    table_ref = f"{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}"
    disposition = WriteDisposition.WRITE_TRUNCATE if BQ_WRITE_MODE == "replace" else WriteDisposition.WRITE_APPEND
    job = client.load_table_from_dataframe(combined, table_ref, job_config=LoadJobConfig(write_disposition=disposition))
    job.result()
    print(f"โหลด {len(combined)} แถว เข้า {table_ref} เรียบร้อย ({BQ_WRITE_MODE})")
else:
    print("ENABLE_BQ_LOAD = False อยู่ — ข้ามขั้นตอนนี้ไปก่อน")
